In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os
import pyarrow
from pathlib import Path
import scipy
from scipy import stats
from scipy.stats import pearsonr, spearmanr

### Alcance y objetivos del recomendador

Este recomendador tiene un objetivo principalmente formativo y exploratorio dentro del proyecto. La implementación se centra en comprender el funcionamiento básico de un sistema de recomendación item-item, sin abordar aspectos propios de un entorno productivo como el despliegue mediante API, model serving, inferencia online o evaluación específica del sistema de recomendaciones.

La sección se mantiene como una extensión del análisis de datos realizado en el proyecto y no forma parte de su pipeline analítico principal.

In [22]:
df_event_session = pd.read_parquet("../data/processed/df_event_session_final.parquet")
product_metrics = pd.read_parquet("../data/processed/product_metrics.parquet")

Incluir solo productos en P90 revenue, P90 PR y P90 engagement al recomendador para simplificar el computo de datos

In [55]:
product_metrics.columns

Index(['product_id', 'engaged_sessions', 'view_sessions', 'cart_sessions',
       'purchase_sessions', 'purchase_events', 'revenue',
       'product_session_purchase_rate', 'observed_view_coverage',
       'revenue_per_engaged_session', 'precio_medio'],
      dtype='str')

In [23]:
#actualizamos variable price
df_event_session['price'] = df_event_session['price'].where(df_event_session['event'] == 'purchase', 0)


In [56]:
top_products = (
    product_metrics.loc[
        product_metrics["engaged_sessions"].ge(product_metrics["engaged_sessions"].quantile(0.9)) |
        product_metrics.revenue.ge(product_metrics.revenue.quantile(0.9)) |
        product_metrics.purchase_events.ge(product_metrics["purchase_events"].quantile(0.9))
    ]
    .product_id.drop_duplicates().to_list()
)

In [57]:
print(product_metrics.product_id.size)
print(len(top_products))
print(f"{len(top_products) / len(product_metrics) * 100} % Productos Incluidos")

45327
7379
16.279480221501533 % Productos Incluidos


In [112]:
#matriz antes del pivot table
matriz_usuario_item = (
    df_event_session.loc[df_event_session['product_id'].isin(top_products) & df_event_session['event'].eq('purchase'),    
                        ['user_id','product_id']]
                        .drop_duplicates()
)

In [113]:
#matriz final
matriz_usuario_item = matriz_usuario_item.pivot_table(index='user_id', columns='product_id', aggfunc='size', fill_value=0)

In [118]:
matriz_usuario_item.shape

(10581, 7284)

Se va a crear un **sistema de recomendacion item-item con la distancia euclidea** como medida de similaridad.

1. Creacion de la matriz Item-Item

In [115]:
from sklearn.metrics.pairwise import euclidean_distances

#Trasponer matriz para que los productos sean las filas
matriz_item_usuario = matriz_usuario_item.T

#Calcula la matriz de distancias euclidianas entre productos
distancias = euclidean_distances(matriz_item_usuario)

#Convierte a dataframe para mejorar visualizacion
distancia_item_item = pd.DataFrame(distancias, index=matriz_item_usuario.index, columns=matriz_item_usuario.index)

#Guarda la matriz de similaridad item-item como parquet
distancia_item_item.to_parquet("../data/processed/distancia_item_item.parquet")

In [123]:
distancia_item_item.shape

(7284, 7284)

In [116]:
distancia_item_item.head(3)

product_id,3762,3763,3774,3776,3806,3928,3936,3945,3959,3978,...,5926128,5926679,5926684,5926686,5926715,5928625,5929649,5929903,5930316,5931773
product_id,,,,,,,,,,,,,,,,,,,,,
3762,0.000000,5.477226,5.385165,5.477226,5.385165,5.744563,5.916080,5.830952,5.477226,6.633250,...,5.477226,5.830952,5.830952,5.830952,5.830952,5.385165,5.385165,5.385165,5.567764,5.385165
3763,5.477226,0.000000,3.000000,2.000000,1.732051,2.645751,3.000000,2.828427,2.828427,4.242641,...,2.000000,2.828427,2.828427,2.828427,2.828427,1.732051,1.732051,1.732051,2.236068,1.732051
3774,5.385165,3.000000,0.000000,3.000000,2.828427,3.162278,3.741657,3.605551,3.000000,4.582576,...,3.000000,3.605551,3.605551,3.605551,3.605551,2.828427,2.828427,2.828427,3.162278,2.828427


2.1. Funcion de entrada del recomendador **(1 input)**: va recomendar 5 productos con mas afinidad al producto input. n puede modificarse a posterior.

In [119]:
def recomendar_productos(product_id, matriz_similaridad, n=5):
    """" 
    Funcion que retorna los n productos mas similares a un producto dado
    """
    if product_id not in matriz_similaridad.index:
        raise ValueError(f"El producto {product_id} no se encuentra en la matriz de similaridad.")
    
    #Ordena por distancia creciente (más parecido)
    similares = matriz_similaridad.loc[product_id].sort_values()

    #El primero será el propio producto asi que lo excluimos
    similares = similares.iloc[1:n+1]

    return similares.index.to_list()

Ejemplo de uso

In [120]:
#Cargamos matriz en memoria
matriz_similaridad = pd.read_parquet("../data/processed/distancia_item_item.parquet")

#Elegimos un producto cualquiera
product_id = 34767

#Recomendamos 5 productos
recomendaciones = recomendar_productos(product_id, matriz_similaridad, n=5)
print(recomendaciones)

[5844415, 5778933, 5826655, 5912597, 5912604]


2.2. Funcion de entrada del recomendador **(Multiples inputs)**: esta es una **versión mejorada** de la 1a función y va a devolver los n productos más similares basandose en x productos de entrada.

In [121]:
def recomendar_multiples_productos(product_ids, matriz_similaridad, n=5):
    """" 
    Funcion que retorna los n productos mas similares a varios productos en conjunto de entrada.
    """
    productos_validos = [p for p in product_ids if p in matriz_similaridad.index]

    if not productos_validos:

        raise ValueError(f"Los productos {product_ids} no se encuentran en la matriz de similaridad.")
    
    #Calcula la distancia media de cada producto respecto a los productos de entrada
    distancias = matriz_similaridad.loc[productos_validos].mean(axis=0)

    #Excluir productos de entrada
    distancias = distancias.drop(productos_validos)
    
    #Ordena por distancia creciente (más parecido) y devuelve los n mas similares
    similares = distancias.sort_values().iloc[:n]

    return similares.index.to_list()

# Los productos de entrada (product_ids, deben ser una lista)

Ejemplo de uso

In [122]:
#Cargamos matriz en memoria
matriz_similaridad = pd.read_parquet("../data/processed/distancia_item_item.parquet")

#Elegimos una lista de productos
product_ids = [5897274, 5912306, 5860250, 5840224, 5832269]

#Recomendamos 5 productos
recomendaciones = recomendar_multiples_productos(product_ids, matriz_similaridad, n=5)
print(recomendaciones)

[5885321, 5897272, 5897273, 5912604, 5928625]
